In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import math
import os
import re
import time

import numpy as np
import psutil
import torch

import fishyrl as frl

# Load Model

In [ ]:
# Load config
cfg = frl.utilities.load_config('../examples/configs/RLGym.yaml', '../examples/configs/Dreamer_12m.yaml', '../examples/configs/Dreamer_Base.yaml')  # ()

# Checkpoint file if not starting from scratch
checkpoint_file = '../examples/runs/RLGym_2026-04-27_19-29-01/checkpoint_0500000.pt'
start_environment_step = int(re.search(r'checkpoint_(\d+).pt', checkpoint_file).group(1)) if checkpoint_file is not None else 0

# Load environment and actions
envs = frl.dreamer.construct_envs(cfg=cfg)
actions = frl.dreamer.construct_actions(cfg=cfg)

# Construct models
world_model, actor_critic_model, utility_modules = frl.dreamer.construct_models(
    # env_obs_dim=math.prod(envs.obs_shape[1:]),  # TODO: Maybe readd default env params
    env_actions=actions,
    device='cuda',
    cfg=cfg,
)

# Load model and buffer
if checkpoint_file is not None:
    frl.dreamer.load_models(
        path=checkpoint_file,
        world_model=world_model,
        actor_critic_model=actor_critic_model,
        utility_modules=utility_modules,
    )

# Simulate

Make sure that you have the v0.8.2 binary (compatibile with `rlviser-py`) for `rlviser` (Linux/MacOS) or `rlviser.exe` (Windows) in the examples folder, as well as a folder `examples/collision_meshes`, extracted using [RLArenaCollisionDumper](https://github.com/ZealanL/RLArenaCollisionDumper).

In the generated `settings.txt`, you may also want to set `camera_state={"Director":0}` for automated camera switching.

In [ ]:
# Use automatic recording?
automate_recording = True

In [ ]:
# Initialize virtual display if running headless
dsp = None
if os.environ.get('DISPLAY') is None:
    import pyvirtualdisplay
    dsp = pyvirtualdisplay.Display(visible=True, size=(1280, 720), backend='xvfb', use_xauth=True).start()

    # Warn if not monitor 0
    if dsp.display != 0:
        print(f'Warning: Display ID is {dsp.display}, but expected 0. If you are running headless, '
            f'you may need to kill all other Xvfb processes with `pkill -f Xvfb`.')

# Take screenshot
def screenshot() -> np.ndarray:
    """Take a screenshot of the primary monitor and return it as a numpy array."""
    import mss
    with mss.mss() as sct:
        frame = np.array(sct.grab(sct.monitors[0]))  # Main monitor
        frame = frame[:, :, [2, 1, 0]]  # Convert from BGRA to RGB
    return frame

In [ ]:
# Create environment
envs = envs.copy(num_envs=1, team_size=2, allow_rendering=True)
obs, _ = envs.reset()
actions = posteriors = hidden_states = initializations = None

# Get device
device = world_model.parameters().__next__().device

# Parameters
open_delay = 5  # Seconds to wait for the RLViser window to open
frame_delay = 4  # Frames to wait after sending packets before starting to record, lines up with RLViser buffer size
speedup = 2  # Speedup factor for rendering

# Loop until done
opened = False
frames = []
while not envs._ended[0]:
    # Render
    envs.render(speedup=speedup)

    # Wait for window to open
    if not opened:
        # Scan for rlviser process
        while 'rlviser' not in [proc.name() for proc in psutil.process_iter()]:
            time.sleep(1)

        # Give time for the window to fully open and set flag
        time.sleep(open_delay)
        opened = True

        # Exit from options menu
        if automate_recording:
            import pyautogui
            pyautogui.click(245, 35)  # Exit the options menu

    # Get frames after frame delay
    if automate_recording and envs._envs[0].renderer.packet_id > frame_delay:
        frames.append(screenshot())

    # Compute action
    ret = frl.dreamer.compute_actions(
        obs=torch.from_numpy(obs).to(device),
        world_model=world_model,
        actor_critic_model=actor_critic_model,
        actions=actions,
        posteriors=posteriors,
        hidden_states=hidden_states,
        initializations=initializations,
    )
    env_actions = ret['env_actions']
    actions = ret['actions']
    posteriors = ret['posterior']
    hidden_states = ret['hidden_state']

    # Step environment
    obs, reward, terminated, truncated, info = envs.step(env_actions)
    initializations = torch.from_numpy(terminated | truncated).to(dtype=bool, device=device)

# Get remaining frames
for _ in range(frame_delay):
    time.sleep(envs._tick_duration)
    if automate_recording:
        frames.append(screenshot())

# Stack frames
frames = np.stack(frames, axis=0)

In [ ]:
# Save video to file with appropriate postfix
step_class = max(0, min(4, math.log10(start_environment_step) // 3)) if start_environment_step > 0 else 0
step_fac_dict = {
    0: (10**0, ''), 1: (10**3, 'k'), 2: (10**6, 'm'), 3: (10**9, 'b'), 4: (10**12, 't')}
step_fac, step_post = step_fac_dict[step_class]
path = f'./images/{cfg.name}_{start_environment_step/1000:.0f}k.mp4'

# Save and display
if automate_recording:
    frl.utilities.export_frames(path, frames, fps=envs.render_fps)
    from IPython.display import display, Video  # noqa: E402, I001
    display(Video(filename=path))

In [ ]:
# Stop virtual display if needed
if dsp is not None:
    dsp.stop()